# PDCP · N10b — Baseline judges and out-of-distribution test
**Notebook 10b of 12.** Kaggle: **GPU T4** (one GPU is enough; P100 also fine) · Internet **On** · **Save & Run All (Commit)**. Expected ≈ 1.5–2 h.

**Inputs:** `kbs-n07-output`, `kbs-n07-v2-output` (LiteJudge data), `kbs-n3-output`, `kbs-n6-part2` (VLM evaluation pairs), `kbs-n09-output` (LiteJudge checkpoints, worker code, evaluation table).

**Question 1 — does PDSG help every kind of judge, not only LiteJudge?** Each baseline is trained twice on the same splits: natural pairs only, and with 30 % balanced PDSG pairs.
| Judge | Input | Idea (prior work it follows) |
|---|---|---|
| Layout-feature MLP | 33 geometric features from node coordinates (quality metrics without stress, edge-length quantiles, node-distance histogram, angular resolution) | discriminator on layout statistics (Klammler, Mchedlidze & Pak, GD 2018 / JGAA 2019) |
| Single-image CNN regressor | image → predicted stress | metric regression from layout images (Haleem et al., IEEE CG&A 2019) |
| CLIP linear probe | frozen CLIP ViT-B/16 embeddings | foundation-model features |
| Pixel statistics | 34 hand-made image statistics (ink amount, spread, grid density) | no learning of features |
| Cue-only (natural only) | the four cue values NU/ELD/XR/MEL | how far the shortcuts alone go |
| LiteJudge (N09 checkpoints, 0 % / 30 %) | image | Siamese CNN (Cai et al. 2021 design = LiteJudge 0 %) |
All judges are written as a per-drawing score s(x); u = s(B) − s(A), so all are antisymmetric and evaluated with the same code as N09.

**Question 2 — does it generalise beyond PDSG drawings?** New test set: 16 graphs not used anywhere (classic real graphs — karate club, Les Misérables, Florentine families, Davis women, dodecahedron, Heawood, Tutte, Frucht — unseen families — hexagonal lattice, circular ladder, random geometric, Holme–Kim — and the training families at a new size n = 75),
each drawn by 10 standard layouts (Kamada–Kawai, Fruchterman–Reingold ×2, rough FR, spectral, circular, KK + noise ×2, FR + noise, random), rendered in 3 styles (training style, thin/coloured, bold/coloured). All 45 pairs per graph × style.

**Outputs** (`/kaggle/working/n10b_output/`): `b_judges.csv`, `b_judges_seeds.csv`, `b_ood_by_style.csv`, `b_ood_pairs.parquet`, `b_scores.parquet`, `n10b_report.json`, figures `fig_b_*.png`, `MANIFEST.txt`.

In [1]:
# ==== SETTINGS ====
N07_ROOT = "/kaggle/input/datasets/resadussalehinrafsan/kbs-n07-output"
N07B_ROOT = "/kaggle/input/datasets/resadussalehinrafsan/kbs-n07-v2-output"
N03_ROOT = "/kaggle/input/datasets/resadussalehinrafsan/kbs-n3-output"
N06_ROOT = "/kaggle/input/datasets/resadussalehinrafsan/kbs-n6-part2"
N09_ROOT = "/kaggle/input/datasets/resadussalehinrafsan/kbs-n09-output"
OUTPUT_DIR = "/kaggle/working/n10b_output"
CACHE = "/tmp/b_cache"
IMG = 224
IZ = 0.02
MIX = 0.30                          # PDSG share for the "+PDSG" variant of every judge
MLP_SEEDS = [0, 1, 2]
MLP_EPOCHS = 40
CNN_EPOCHS = 15
CNN_EPOCH_IMAGES = 8000
EPOCH_PAIRS = 6000
TTA = 8
OOD_SEED = 2027
OOD_STYLES = {"train": {}, "thin": {"node_r": 3.0, "edge_w": 0.7, "node_fill": "#1f77b4", "edge_col": "#555555"},
              "bold": {"node_r": 9.0, "edge_w": 2.5, "node_fill": "#d62728", "edge_col": "#000000"}}
N_CPU = 4
PRETRAINED = True
FAST = False

In [2]:
import os, sys, json, time, glob, shutil, subprocess, importlib, itertools
T0 = time.time()
os.makedirs(OUTPUT_DIR, exist_ok=True); os.makedirs(CACHE, exist_ok=True)
for p in ["timm", "joblib", "pyarrow", "sklearn"]:
    try: importlib.import_module(p)
    except ImportError: subprocess.run([sys.executable, "-m", "pip", "install", "-q", "scikit-learn" if p == "sklearn" else p], check=True)
import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F, networkx as nx, matplotlib
matplotlib.use("Agg"); import matplotlib.pyplot as plt
from joblib import Parallel, delayed
from sklearn.linear_model import LogisticRegression
DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu"); AMP = DEV.type == "cuda"
assert AMP or FAST, "Set Accelerator = GPU"
def find(root, name):
    f = sorted(glob.glob(f"{root}/**/{name}", recursive=True)); assert f, f"{name} not found under {root}"; return os.path.dirname(f[0])
N07, N07B, N03, N06 = find(N07_ROOT, "lj_drawings.parquet"), find(N07B_ROOT, "lj_pairs_full.parquet"), find(N03_ROOT, "pairs.parquet"), find(N06_ROOT, "mel_cf_pairs.parquet")
N09 = find(N09_ROOT, "eval_pairs_meta.parquet")
for f in ["pdcp_utils.py", "lj_train.py"]: shutil.copy(f"{N09}/{f}" if os.path.exists(f"{N09}/{f}") else f"{N07}/{f}", f"{OUTPUT_DIR}/{f}")
sys.path.insert(0, OUTPUT_DIR); import pdcp_utils as U; import lj_train as LT
print("N09:", N09, "| utils", U.__version__, "| device", DEV)
D7 = pd.read_parquet(f"{N07}/lj_drawings.parquet").assign(root=N07)
D7b = pd.read_parquet(f"{N07B}/lj_drawings_new.parquet").assign(root=N07B)
LJP = pd.read_parquet(f"{N07B}/lj_pairs_full.parquet")
PR = pd.read_parquet(f"{N03}/pairs.parquet"); DR = pd.read_parquet(f"{N03}/drawings.parquet")
MCP = pd.read_parquet(f"{N06}/mel_cf_pairs.parquet"); MCD = pd.read_parquet(f"{N06}/mel_cf_drawings.parquet")
PR = PR[PR.set.isin(["main", "e1b_mooney", "counterfactual"])]
EVM = pd.read_parquet(f"{N09}/eval_pairs_meta.parquet")
if FAST:
    LJP = LJP[LJP.graph.isin([0, 18, 21])]
    PR = pd.concat([PR[PR.set == s].head(60) for s in ["main", "e1b_mooney"]] + [PR[(PR.set == "counterfactual") & PR.base_pair.isin(PR[PR.set == "main"].head(60).pair_id)]])
    MCP = MCP[MCP.base_pair.isin(PR.pair_id)]
    MLP_SEEDS, MLP_EPOCHS, CNN_EPOCHS, CNN_EPOCH_IMAGES, EPOCH_PAIRS, TTA = [0], 2, 1, 256, 256, 2

N09: /kaggle/input/datasets/resadussalehinrafsan/kbs-n09-output/n09_output | utils 0.1.3 | device cuda


## 1. Image registry (identical to N09, so pair indices match) + layout features from coordinates

In [3]:
M5 = ["ksm", "nu", "eld", "xr", "mel"]
need7 = set(LJP.drawing_A) | set(LJP.drawing_B)
R7 = pd.concat([D7, D7b]); R7 = R7[R7.drawing_id.isin(need7)].assign(src="png", path=lambda d: d.root + "/" + d.png)
need3 = set(PR.drawing_A) | set(PR.drawing_B) | set(MCP.drawing_B)
R3 = DR[DR.drawing_id.isin(need3)].assign(src="render", mel=np.nan)
R6 = MCD[MCD.drawing_id.isin(set(MCP.drawing_A))].assign(src="render")
REG = pd.concat([R7, R3, R6], ignore_index=True).drop_duplicates("drawing_id").reset_index(drop=True)

def layout_feats(X, gc):
    """33 geometric features of a drawing; graph-theoretic distances are NOT used, so stress cannot be computed from them."""
    X = U.normalise(X); n = len(X); E = gc.E
    m = U.all_metrics(X, gc, with_realism=True)
    L = np.linalg.norm(X[E[:, 0]] - X[E[:, 1]], axis=1); mu = L.mean() + 1e-12
    q = np.quantile(L / mu, [0.1, 0.25, 0.5, 0.75, 0.9])
    Dm = np.linalg.norm(X[:, None] - X[None], axis=-1)
    h = np.histogram(Dm[np.triu_indices(n, 1)], bins=16, range=(0, 1.4143))[0] / (n * (n - 1) / 2)
    nn_ = (Dm + np.eye(n) * 10).min(1) * np.sqrt(n)
    nb = [[] for _ in range(n)]
    for a, b in E: nb[a].append(b); nb[b].append(a)
    ang = []
    for i in range(n):
        if len(nb[i]) >= 2:
            v = X[nb[i]] - X[i]; t = np.sort(np.arctan2(v[:, 1], v[:, 0]))
            ang.append(np.diff(np.r_[t, t[0] + 2 * np.pi]).min() / (2 * np.pi / len(t)))
    return np.r_[[m[k] for k in ["nu", "eld", "xr", "mel", "aspect", "nn_max", "nn_min", "dense"]], q, L.max() / mu, h,
                 nn_.mean(), nn_.std() / (nn_.mean() + 1e-12), np.mean(ang) if ang else 1.0].astype(np.float32), m
N_FEAT = 33

def load_one(rec):
    import sys; sys.path.insert(0, OUTPUT_DIR)
    import numpy as np, pdcp_utils as U
    from PIL import Image
    gc = U.GraphCache(U.make_graph(rec["family"], int(rec["n"]), int(rec["graph_seed"])))
    X = np.array(rec["coords"]).reshape(-1, 2)
    f, m = layout_feats(X, gc)
    im = U.render_png(gc, X, None, res=384) if rec["src"] == "render" else Image.open(rec["path"])
    return np.asarray(im.convert("L").resize((IMG, IMG), Image.LANCZOS), dtype=np.uint8), f, float(m["mel"])
recs = REG.reindex(columns=["drawing_id", "src", "path", "family", "n", "graph_seed", "coords", "mel"]).to_dict("records")
if os.path.exists(f"{CACHE}/reg_imgs.npy") and len(np.load(f"{CACHE}/reg_feats.npy")) == len(REG):
    print("cache exists")
else:
    out = Parallel(n_jobs=N_CPU, batch_size=32, verbose=1)(delayed(load_one)(r) for r in recs)
    np.save(f"{CACHE}/reg_imgs.npy", np.stack([o[0] for o in out])); np.save(f"{CACHE}/reg_feats.npy", np.stack([o[1] for o in out]))
    REG["mel"] = REG["mel"].where(REG["mel"].notna(), pd.Series([o[2] for o in out]))
    np.save(f"{CACHE}/reg_met.npy", REG[M5].to_numpy(np.float32))
REG_IMG = np.load(f"{CACHE}/reg_imgs.npy", mmap_mode="r"); REG_F = np.load(f"{CACHE}/reg_feats.npy"); MET = np.load(f"{CACHE}/reg_met.npy")
assert np.isfinite(REG_F).all() and np.isfinite(MET).all()
print(len(REG), "drawings |", REG_F.shape, "features |", f"{(time.time()-T0)/60:.1f} min")

[Parallel(n_jobs=4)]: Using backend LokyBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done 1352 tasks      | elapsed:   19.0s
[Parallel(n_jobs=4)]: Done 6152 tasks      | elapsed:  1.3min
[Parallel(n_jobs=4)]: Done 14152 tasks      | elapsed:  4.0min
[Parallel(n_jobs=4)]: Done 16117 out of 16117 | elapsed:  4.8min finished


16117 drawings | (16117, 33) features | 5.2 min


## 2. Training pairs and evaluation pairs (checked against N09)

In [4]:
IDX = dict(zip(REG.drawing_id, range(len(REG))))
def with_idx(df):
    df = df.copy(); df["A_idx"] = df.drawing_A.map(IDX).astype(int); df["B_idx"] = df.drawing_B.map(IDX).astype(int)
    df["d_ksm"] = MET[df.B_idx, 0] - MET[df.A_idx, 0]; return df
L = with_idx(LJP)
TRAIN = L[(L.split == "train") & ((L.kind == "natural") | L.in_bal)][["pair_id", "kind", "A_idx", "B_idx", "d_ksm"]].reset_index(drop=True)
EV = EVM[EVM.pair_id.isin(set(L.pair_id) | set(PR.pair_id) | set(MCP.pair_id))].copy()
chk = pd.concat([L[["pair_id", "drawing_A", "drawing_B"]], PR[["pair_id", "drawing_A", "drawing_B"]], MCP[["pair_id", "drawing_A", "drawing_B"]]]).drop_duplicates("pair_id").set_index("pair_id")
a_, b_ = EV.pair_id.map(chk.drawing_A).map(IDX).values, EV.pair_id.map(chk.drawing_B).map(IDX).values
if FAST: EV["A_idx"], EV["B_idx"] = a_.astype(int), b_.astype(int)
assert (EV.A_idx.values == a_).all() and (EV.B_idx.values == b_).all(), "registry order differs from N09"
VAL = EV[EV.eval_set.isin(["lj_nat_val", "lj_pdsg_val"])]
print("train", TRAIN.kind.value_counts().to_dict(), "| eval", EV.eval_set.value_counts().to_dict())

train {'natural': 7573, 'pdsg': 3983} | eval {'n03_cf': 2839, 'n03_main': 1926, 'n06_cf': 1546, 'lj_nat_val': 1316, 'lj_nat_test': 1282, 'lj_pdsg_test': 721, 'lj_pdsg_val': 700, 'n03_e1b': 289, 'lj_pdsg_extra_test': 183}


## 3. Out-of-distribution test set (new graphs × standard layout algorithms × 3 rendering styles)

In [5]:
def ood_graphs():
    G = {"karate": nx.karate_club_graph(), "lesmis": nx.les_miserables_graph(), "florentine": nx.florentine_families_graph(),
         "davis": nx.davis_southern_women_graph(), "dodecahedral": nx.dodecahedral_graph(), "heawood": nx.heawood_graph(),
         "tutte": nx.tutte_graph(), "frucht": nx.frucht_graph(), "hexlattice": nx.hexagonal_lattice_graph(4, 4),
         "circladder": nx.circular_ladder_graph(20), "rgg60": nx.random_geometric_graph(60, 0.22, seed=11),
         "holmekim60": nx.powerlaw_cluster_graph(60, 2, 0.3, seed=12), "er75": nx.gnp_random_graph(75, 3.5 / 74, seed=13),
         "ba75": nx.barabasi_albert_graph(75, 2, seed=14), "ws75": nx.connected_watts_strogatz_graph(75, 4, 0.1, seed=15),
         "tree75": U.make_graph("TREE", 75, 16)}
    out = {}
    for k, g in G.items():
        g = nx.Graph(g); g.remove_edges_from(nx.selfloop_edges(g)); g = g.subgraph(max(nx.connected_components(g), key=len)).copy()
        out[k] = nx.convert_node_labels_to_integers(g)
    return out
OG = ood_graphs()
if FAST: OG = {k: OG[k] for k in ["karate", "dodecahedral"]}
def layouts(G, seed):
    n = len(G); rng = np.random.default_rng(seed); arr = lambda p: U.normalise(np.array([p[i] for i in range(n)], dtype=float))
    kk = arr(nx.kamada_kawai_layout(G)); fr = arr(nx.spring_layout(G, seed=seed))
    return {"kk": kk, "fr_a": fr, "fr_b": arr(nx.spring_layout(G, seed=seed + 1)), "fr_rough": arr(nx.spring_layout(G, seed=seed + 2, iterations=15)),
            "spectral": arr(nx.spectral_layout(G)), "circular": arr(nx.circular_layout(G)),
            "kk_noise05": U.normalise(kk + rng.normal(0, 0.05, kk.shape)), "kk_noise15": U.normalise(kk + rng.normal(0, 0.15, kk.shape)),
            "fr_noise10": U.normalise(fr + rng.normal(0, 0.10, fr.shape)), "random": rng.uniform(0, 1, (n, 2))}
def ood_one(gname, lname, X, style, edges, n):
    import sys; sys.path.insert(0, OUTPUT_DIR)
    import numpy as np, networkx as nx, pdcp_utils as U
    G = nx.Graph(); G.add_nodes_from(range(n)); G.add_edges_from(edges); gc = U.GraphCache(G)
    f, m = layout_feats(X, gc)
    im = U.render_png(gc, X, None, res=384, **OOD_STYLES[style]).convert("L").resize((IMG, IMG))
    return {"gname": gname, "layout": lname, "style": style, "n": n, **{k: m[k] for k in M5}}, np.asarray(im, dtype=np.uint8), f
jobs = []
for gi, (gname, G) in enumerate(OG.items()):
    for lname, X in layouts(G, OOD_SEED + gi).items():
        for st in OOD_STYLES: jobs.append((gname, lname, X, st, list(G.edges()), len(G)))
res = Parallel(n_jobs=N_CPU, batch_size=8)(delayed(ood_one)(*j) for j in jobs)
OD = pd.DataFrame([r[0] for r in res]); OOD_IMG = np.stack([r[1] for r in res]); OOD_F = np.stack([r[2] for r in res])
OD["oidx"] = range(len(OD))
op = []
for (g, st), d in OD.groupby(["gname", "style"]):
    for a, b in itertools.combinations(d.itertuples(), 2):
        op.append({"pair_id": f"O_{g}_{st}_{a.layout}__{b.layout}", "gname": g, "style": st, "layout_A": a.layout, "layout_B": b.layout,
                   "A_oidx": a.oidx, "B_oidx": b.oidx, "A_ksm": a.ksm, "B_ksm": b.ksm, **{f"d_{k}": getattr(b, k) - getattr(a, k) for k in M5}})
OODP = pd.DataFrame(op)
OD[M5] = OD[M5].astype(float); OODP[[c for c in OODP if c.startswith(("d_", "A_ksm", "B_ksm"))]] = OODP[[c for c in OODP if c.startswith(("d_", "A_ksm", "B_ksm"))]].astype(float)
print(len(OG), "graphs |", len(OD), "OOD drawings |", len(OODP), "OOD pairs | stress range", round(float(OD.ksm.min()), 3), "-", round(float(OD.ksm.max()), 3))
print({k: round(float(v), 3) for k, v in OD.groupby("layout").ksm.median().items()})

16 graphs | 480 OOD drawings | 2160 OOD pairs | stress range 0.062 - 0.702
{'circular': 0.397, 'fr_a': 0.299, 'fr_b': 0.312, 'fr_noise10': 0.365, 'fr_rough': 0.359, 'kk': 0.267, 'kk_noise05': 0.286, 'kk_noise15': 0.364, 'random': 0.423, 'spectral': 0.34}


## 4. Judges

In [6]:
SCORES = {}                                       # judge name -> (score per registry drawing, score per OOD drawing)
NAT = TRAIN[TRAIN.kind == "natural"]; PDS = TRAIN[TRAIN.kind == "pdsg"]
def labels(d): return np.where(d > IZ, 2, np.where(d < -IZ, 0, 1))
def epoch_pairs(rng, p):
    n_p = int(round(p * EPOCH_PAIRS)); n_n = EPOCH_PAIRS - n_p; parts = []
    if n_n: parts.append(NAT.iloc[rng.choice(len(NAT), n_n, replace=n_n > len(NAT))])
    if n_p: parts.append(PDS.iloc[rng.choice(len(PDS), n_p, replace=n_p > len(PDS))])
    E = pd.concat(parts).sample(frac=1, random_state=int(rng.integers(1e9))); y = labels(E.d_ksm.values)
    sw = rng.random(len(E)) < 0.5
    return np.where(sw, E.B_idx, E.A_idx), np.where(sw, E.A_idx, E.B_idx), np.where(sw, 2 - y, y)
def val_acc(s):
    u = s[VAL.B_idx.values] - s[VAL.A_idx.values]; m = VAL.d_ksm.abs().values >= 0.02
    return {k: round(float(((u > 0) == (VAL.d_ksm.values > 0))[m & (VAL.eval_set == k).values].mean()), 3) for k in ["lj_nat_val", "lj_pdsg_val"]}

# ---- 4a. LiteJudge (N09 checkpoints) -------------------------------------------------------------
def score_imgs(model, imgs, tta=TTA, bs=128):
    model.eval(); out = np.zeros(len(imgs)); tr = [(k, f) for f in (0, 1) for k in range(4)][:tta]
    with torch.no_grad():
        for s in range(0, len(imgs), bs):
            x = torch.from_numpy(np.ascontiguousarray(imgs[s:s + bs])).to(DEV).unsqueeze(1).float() / 255; acc = 0
            for k, f in tr:
                with torch.autocast("cuda", dtype=torch.float16, enabled=AMP):
                    sc = model(LT.dihedral(x, k, f)); sc = sc[0] if isinstance(sc, tuple) else sc
                acc = acc + sc.float().view(-1)
            out[s:s + bs] = (acc / len(tr)).cpu().numpy()
    return out
for p, ck in [(0.0, "cnx_p000_s0"), (0.3, "cnx_p030_s0")]:
    f = f"{N09}/ckpt/{ck}.pt"
    if not os.path.exists(f): print("missing checkpoint", f); continue
    m = LT.LiteJudge("convnext_t", IMG, pretrained=False).to(DEV); m.load_state_dict({k: v.float() for k, v in torch.load(f, map_location=DEV).items()})
    SCORES[f"LiteJudge|{p}"] = (score_imgs(m, REG_IMG), score_imgs(m, OOD_IMG)); print("LiteJudge", p, val_acc(SCORES[f"LiteJudge|{p}"][0]))
    del m; torch.cuda.empty_cache()

# ---- 4b. Layout-feature MLP (Klammler-style discriminator on coordinates) ---------------------------
mu_f, sd_f = REG_F[np.unique(np.r_[TRAIN.A_idx, TRAIN.B_idx])].mean(0), REG_F[np.unique(np.r_[TRAIN.A_idx, TRAIN.B_idx])].std(0) + 1e-6
FZ = torch.tensor((REG_F - mu_f) / sd_f, dtype=torch.float32, device=DEV); FZo = torch.tensor((OOD_F - mu_f) / sd_f, dtype=torch.float32, device=DEV)
class MLP(nn.Module):
    def __init__(s, d):
        super().__init__(); s.f = nn.Sequential(nn.Linear(d, 128), nn.GELU(), nn.Dropout(0.1), nn.Linear(128, 128), nn.GELU(), nn.Linear(128, 1)); s.t = nn.Parameter(torch.tensor(-1.0))
    def forward(s, x): return s.f(x).squeeze(-1)
    def tau(s): return F.softplus(s.t) + 1e-3
for p in [0.0, MIX]:
    for seed in MLP_SEEDS:
        torch.manual_seed(seed); rng = np.random.default_rng(seed); m = MLP(N_FEAT).to(DEV)
        opt = torch.optim.AdamW(m.parameters(), lr=1e-3, weight_decay=1e-4); sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, MLP_EPOCHS)
        for ep in range(MLP_EPOCHS):
            A, B, y = epoch_pairs(rng, p); m.train()
            for s in range(0, len(A), 256):
                ia, ib = A[s:s + 256], B[s:s + 256]; u = m(FZ[ib]) - m(FZ[ia])
                loss = LT.coral_loss(u, torch.tensor(y[s:s + 256], device=DEV), m.tau()); opt.zero_grad(); loss.backward(); opt.step()
            sch.step()
        m.eval()
        with torch.no_grad(): SCORES[f"LayoutMLP|{p}|s{seed}"] = (m(FZ).cpu().numpy().astype(float), m(FZo).cpu().numpy().astype(float))
        print("LayoutMLP", p, seed, val_acc(SCORES[f"LayoutMLP|{p}|s{seed}"][0]))

# ---- 4c. Single-image CNN stress regressor (Haleem-style) -----------------------------------------
class SmallCNN(nn.Module):
    def __init__(s):
        super().__init__(); L = []; c = 1
        for o in [32, 64, 128, 128, 256]: L += [nn.Conv2d(c, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU(inplace=True), nn.MaxPool2d(2)]; c = o
        s.f = nn.Sequential(*L, nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(256, 128), nn.ReLU(inplace=True), nn.Dropout(0.2), nn.Linear(128, 1))
    def forward(s, x): return s.f(x).squeeze(-1)
nat_imgs = np.unique(np.r_[NAT.A_idx, NAT.B_idx]); pds_imgs = np.unique(PDS.A_idx)
k_mu, k_sd = MET[nat_imgs, 0].mean(), MET[nat_imgs, 0].std()
for p in [0.0, MIX]:
    torch.manual_seed(0); rng = np.random.default_rng(0); gen = torch.Generator().manual_seed(0)
    m = SmallCNN().to(DEV); opt = torch.optim.AdamW(m.parameters(), lr=1e-3, weight_decay=1e-4)
    steps = CNN_EPOCHS * int(np.ceil(CNN_EPOCH_IMAGES / 128)); sch = torch.optim.lr_scheduler.OneCycleLR(opt, 2e-3, total_steps=steps); scaler = torch.amp.GradScaler("cuda", enabled=AMP)
    for ep in range(CNN_EPOCHS):
        n_p = int(round(p * CNN_EPOCH_IMAGES)); ids = np.r_[rng.choice(nat_imgs, CNN_EPOCH_IMAGES - n_p), rng.choice(pds_imgs, n_p)] if n_p else rng.choice(nat_imgs, CNN_EPOCH_IMAGES)
        rng.shuffle(ids); m.train()
        for s in range(0, len(ids), 128):
            b = np.sort(ids[s:s + 128]); x = LT.augment(torch.from_numpy(np.ascontiguousarray(REG_IMG[b])).to(DEV).unsqueeze(1).float() / 255, gen)
            t = torch.tensor((MET[b, 0] - k_mu) / k_sd, dtype=torch.float32, device=DEV)
            with torch.autocast("cuda", dtype=torch.float16, enabled=AMP): loss = F.mse_loss(m((x - 0.5) / 0.5).float(), t)
            opt.zero_grad(); scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sch.step()
    class Wrap(nn.Module):
        def __init__(s, m): super().__init__(); s.m = m
        def forward(s, x): return s.m((x - 0.5) / 0.5)
    SCORES[f"ImageCNN|{p}"] = (score_imgs(Wrap(m), REG_IMG), score_imgs(Wrap(m), OOD_IMG)); print("ImageCNN", p, val_acc(SCORES[f"ImageCNN|{p}"][0]))
    del m; torch.cuda.empty_cache()
print(f"{(time.time()-T0)/60:.1f} min")

/tmp/ipykernel_24/1595619020.py:20: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  x = torch.from_numpy(np.ascontiguousarray(imgs[s:s + bs])).to(DEV).unsqueeze(1).float() / 255; acc = 0


LiteJudge 0.0 {'lj_nat_val': 0.999, 'lj_pdsg_val': 0.848}
LiteJudge 0.3 {'lj_nat_val': 1.0, 'lj_pdsg_val': 0.989}
LayoutMLP 0.0 0 {'lj_nat_val': 0.998, 'lj_pdsg_val': 0.762}
LayoutMLP 0.0 1 {'lj_nat_val': 0.998, 'lj_pdsg_val': 0.81}
LayoutMLP 0.0 2 {'lj_nat_val': 0.998, 'lj_pdsg_val': 0.798}
LayoutMLP 0.3 0 {'lj_nat_val': 0.997, 'lj_pdsg_val': 0.958}
LayoutMLP 0.3 1 {'lj_nat_val': 0.997, 'lj_pdsg_val': 0.964}
LayoutMLP 0.3 2 {'lj_nat_val': 0.997, 'lj_pdsg_val': 0.96}


/tmp/ipykernel_24/1595619020.py:76: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  opt.zero_grad(); scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sch.step()


ImageCNN 0.0 {'lj_nat_val': 0.991, 'lj_pdsg_val': 0.714}
ImageCNN 0.3 {'lj_nat_val': 0.995, 'lj_pdsg_val': 0.907}
19.9 min


In [7]:
# ---- 4d. Linear judges on fixed per-drawing features: pixel statistics, CLIP embeddings, cue values ----------
def pix_feats(imgs, bs=2048):
    out = []
    H = imgs.shape[1]; yy, xx = np.mgrid[0:H, 0:H] / H
    for s in range(0, len(imgs), bs):
        ink = 1 - np.asarray(imgs[s:s + bs], dtype=np.float32) / 255; tot = ink.sum((1, 2)) + 1e-6
        cx, cy = (ink * xx).sum((1, 2)) / tot, (ink * yy).sum((1, 2)) / tot
        r = np.sqrt((xx[None] - cx[:, None, None]) ** 2 + (yy[None] - cy[:, None, None]) ** 2)
        rm = (ink * r).sum((1, 2)) / tot; rs = np.sqrt((ink * (r - rm[:, None, None]) ** 2).sum((1, 2)) / tot)
        k = H // 8; g8 = ink[:, :k * 8, :k * 8].reshape(len(ink), 8, k, 8, k).mean((2, 4)).reshape(len(ink), -1)
        k4 = H // 4; g4 = ink[:, :k4 * 4, :k4 * 4].reshape(len(ink), 4, k4, 4, k4).mean((2, 4)).reshape(len(ink), -1)
        g4 = g4 / (g4.sum(1, keepdims=True) + 1e-6)
        rows, cols = (ink.max(2) > 0.3), (ink.max(1) > 0.3)
        bb = rows.sum(1) * cols.sum(1) / H ** 2
        out.append(np.c_[ink.mean((1, 2)), (ink > 0.6).mean((1, 2)), ((ink > 0.15) & (ink <= 0.6)).mean((1, 2)), rm, rs,
                         g8.std(1) / (g8.mean(1) + 1e-6), g8.max(1) / (g8.sum(1) + 1e-6), (g8 < 0.1 * g8.mean(1, keepdims=True)).mean(1),
                         bb, cx, cy, np.abs(cx - 0.5) + np.abs(cy - 0.5), rs / (rm + 1e-6), (ink > 0.6).mean((1, 2)) / (ink.mean((1, 2)) + 1e-6),
                         ink.std((1, 2)), (r * (ink > 0.6)).sum((1, 2)) / ((ink > 0.6).sum((1, 2)) + 1e-6), rows.mean(1), cols.mean(1), g4])
    return np.concatenate(out).astype(np.float32)
def clip_embed(imgs, model, bs=256):
    cfg = model.pretrained_cfg; mu = torch.tensor(cfg["mean"], device=DEV).view(1, 3, 1, 1); sd = torch.tensor(cfg["std"], device=DEV).view(1, 3, 1, 1)
    out = []
    with torch.no_grad():
        for s in range(0, len(imgs), bs):
            x = torch.from_numpy(np.ascontiguousarray(imgs[s:s + bs])).to(DEV).unsqueeze(1).float().div(255).expand(-1, 3, -1, -1)
            with torch.autocast("cuda", dtype=torch.float16, enabled=AMP): e = model((x - mu) / sd)
            out.append(F.normalize(e.float(), dim=-1).cpu().numpy())
    return np.concatenate(out)
FEATS = {"PixelStats": (pix_feats(REG_IMG), pix_feats(OOD_IMG)), "CueOnly": (MET[:, 1:5].astype(np.float32), OD[["nu", "eld", "xr", "mel"]].to_numpy(np.float32))}
try:
    import timm
    cm = timm.create_model("vit_base_patch16_clip_224.openai", pretrained=PRETRAINED, num_classes=0).to(DEV).eval()
    FEATS["CLIPprobe"] = (clip_embed(REG_IMG, cm), clip_embed(OOD_IMG, cm)); del cm; torch.cuda.empty_cache()
except Exception as e:
    print("CLIP probe skipped:", type(e).__name__, str(e)[:200])
def fit_linear(Fr, p, C):
    sd = Fr[np.unique(np.r_[TRAIN.A_idx, TRAIN.B_idx])].std(0) + 1e-6; Z = Fr / sd
    T = TRAIN[TRAIN.d_ksm.abs() >= IZ] if p > 0 else TRAIN[(TRAIN.kind == "natural") & (TRAIN.d_ksm.abs() >= IZ)]
    X = Z[T.B_idx.values] - Z[T.A_idx.values]; y = (T.d_ksm.values > 0).astype(int)
    w = np.where(T.kind.values == "pdsg", p / (1 - p) * (T.kind == "natural").sum() / max(1, (T.kind == "pdsg").sum()), 1.0)
    lr = LogisticRegression(C=C, fit_intercept=False, max_iter=3000).fit(np.r_[X, -X], np.r_[y, 1 - y], sample_weight=np.r_[w, w])
    return lr.coef_.ravel() / sd
for name, (Fr, Fo) in FEATS.items():
    for p in ([0.0] if name == "CueOnly" else [0.0, MIX]):
        best = None
        for Cc in [0.01, 0.1, 1.0, 10.0]:
            w = fit_linear(Fr, p, Cc); s = Fr @ w; va = val_acc(s); sc = va["lj_nat_val"] + va["lj_pdsg_val"] if p > 0 else va["lj_nat_val"]
            if best is None or sc > best[0]: best = (sc, w, Cc, va)
        SCORES[f"{name}|{p}"] = (Fr @ best[1], Fo @ best[1]); print(name, p, "C =", best[2], best[3])
print(len(SCORES), "judges scored |", f"{(time.time()-T0)/60:.1f} min")

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  599MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

PixelStats 0.0 C = 10.0 {'lj_nat_val': 0.985, 'lj_pdsg_val': 0.67}
PixelStats 0.3 C = 10.0 {'lj_nat_val': 0.976, 'lj_pdsg_val': 0.874}
CueOnly 0.0 C = 0.01 {'lj_nat_val': 0.983, 'lj_pdsg_val': 0.213}
CLIPprobe 0.0 C = 0.1 {'lj_nat_val': 0.996, 'lj_pdsg_val': 0.701}
CLIPprobe 0.3 C = 0.1 {'lj_nat_val': 0.988, 'lj_pdsg_val': 0.96}
15 judges scored | 21.7 min


## 5. Evaluation (same measures as N09) and OOD accuracy

In [8]:
CUES = ["nu", "eld", "xr", "mel"]
def acc(u, d):
    m = np.abs(d) >= 0.02; return float(((u[m] > 0) == (d[m] > 0)).mean()) if m.any() else np.nan
def betas(df, u):
    X = df[["d_" + k for k in ["ksm"] + CUES]].to_numpy(float); X = (X - X.mean(0)) / (X.std(0) + 1e-12); y = (u - u.mean()) / (u.std() + 1e-12)
    return dict(zip(["ksm"] + CUES, np.linalg.lstsq(np.c_[np.ones(len(df)), X], y, rcond=None)[0][1:]))
def share(b): c = sum(abs(b[k]) for k in CUES); return c / (c + abs(b["ksm"]))
def evaluate(s_reg, s_ood):
    r = {}; u_all = s_reg[EV.B_idx.values] - s_reg[EV.A_idx.values]; E2 = EV.assign(u=u_all)
    G = {k: g for k, g in E2.groupby("eval_set")}
    for k in ["lj_nat_test", "lj_pdsg_test", "lj_pdsg_extra_test", "n03_main", "n03_e1b"]:
        if k in G: r["acc_" + k] = acc(G[k].u.values, G[k].d_ksm.values)
    for k in ["lj_pdsg_test", "n03_main"]:
        if k in G and len(G[k]) > 10:
            b = betas(G[k], G[k].u.values); r["cue_share_" + k] = share(b); r.update({f"beta_{q}_{k}": v for q, v in b.items()})
            z = G[k][G[k].dstress_design == 0]; a, c = z[z.cond == "congruent"], z[z.cond == "conflict_all"]
            r["lure_" + k] = float((a.u > 0).mean() - (c.u > 0).mean()) if len(a) and len(c) else np.nan
    base = E2[E2.eval_set == "n03_main"].set_index("pair_id").u
    cf = E2[E2.eval_set.isin(["n03_cf", "n06_cf"]) & E2.base_pair.isin(base.index)]
    fl = (cf.u.values > 0) != (cf.base_pair.map(base).values > 0)
    for px in ["null", "nu", "eld", "xr", "mel"]:
        mk = (cf.cf_proxy == px).values; r[f"flip_{px}"] = float(fl[mk].mean()) if mk.any() else np.nan
    uo = s_ood[OODP.B_oidx.values] - s_ood[OODP.A_oidx.values]; d = OODP.d_ksm.values
    r["acc_ood"] = acc(uo, d)
    for st in OOD_STYLES: r[f"acc_ood_{st}"] = acc(uo[(OODP["style"] == st).values], d[(OODP["style"] == st).values])
    nr = ((OODP.A_ksm <= 0.6) & (OODP.B_ksm <= 0.6)).values; r["acc_ood_natural_range"] = acc(uo[nr], d[nr])
    r["rho_ood"] = float(OD.assign(s=s_ood).groupby(["gname", "style"]).apply(lambda g: g.s.corr(g.ksm, method="spearman")).mean())
    return r, uo
rows, UO = [], {}
for k, (sr, so) in SCORES.items():
    parts = k.split("|"); r, uo = evaluate(sr, so); UO[k] = uo
    rows.append({"judge": parts[0], "pdsg_share": float(parts[1]), "seed": int(parts[2][1:]) if len(parts) > 2 else 0, **r})
JS = pd.DataFrame(rows); JS.round(4).to_csv(f"{OUTPUT_DIR}/b_judges_seeds.csv", index=False)
num = [c for c in JS.columns if c not in ["judge", "pdsg_share", "seed"]]
JM = JS.groupby(["judge", "pdsg_share"])[num].mean().reset_index(); JM.insert(2, "n_seeds", JS.groupby(["judge", "pdsg_share"]).size().values)
order = ["LiteJudge", "ImageCNN", "CLIPprobe", "LayoutMLP", "PixelStats", "CueOnly"]
JM["o"] = JM.judge.map({j: i for i, j in enumerate(order)}); JM = JM.sort_values(["o", "pdsg_share"]).drop(columns="o")
JM.round(4).to_csv(f"{OUTPUT_DIR}/b_judges.csv", index=False)
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 40)
show = ["judge", "pdsg_share", "n_seeds", "acc_lj_nat_test", "acc_lj_pdsg_test", "acc_n03_main", "cue_share_lj_pdsg_test", "cue_share_n03_main",
        "lure_n03_main", "flip_null", "flip_nu", "flip_mel", "acc_ood", "acc_ood_train", "acc_ood_thin", "acc_ood_bold", "acc_ood_natural_range", "rho_ood"]
print(JM[[c for c in show if c in JM]].round(3).to_string(index=False))
OS = pd.DataFrame([{"judge": k, "style": st, "layout_pair_kind": kind, "acc": acc(UO[k][m], OODP.d_ksm.values[m]), "n": int(m.sum())}
                   for k in UO for st in OOD_STYLES for kind, m in [("all", (OODP["style"] == st).values),
                   ("both_natural_range", ((OODP["style"] == st) & (OODP.A_ksm <= 0.6) & (OODP.B_ksm <= 0.6)).values)]])
OS.round(4).to_csv(f"{OUTPUT_DIR}/b_ood_by_style.csv", index=False)
OUT = OODP.copy()
for k in UO: OUT["u_" + k.replace("|", "_")] = UO[k]
OUT.to_parquet(f"{OUTPUT_DIR}/b_ood_pairs.parquet", index=False)
OD.assign(**{"s_" + k.replace("|", "_"): v[1] for k, v in SCORES.items()}).to_parquet(f"{OUTPUT_DIR}/b_scores.parquet", index=False)

/tmp/ipykernel_24/3906115964.py:27: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  r["rho_ood"] = float(OD.assign(s=s_ood).groupby(["gname", "style"]).apply(lambda g: g.s.corr(g.ksm, method="spearman")).mean())
/tmp/ipykernel_24/3906115964.py:27: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  r["rho_ood"] = float(OD.assign(s=s_ood).groupby(["gname", "style"]).apply(lambda g: g.s.corr(g.ksm, method="spearman")).mean())


     judge  pdsg_share  n_seeds  acc_lj_nat_test  acc_lj_pdsg_test  acc_n03_main  cue_share_lj_pdsg_test  cue_share_n03_main  lure_n03_main  flip_null  flip_nu  flip_mel  acc_ood  acc_ood_train  acc_ood_thin  acc_ood_bold  acc_ood_natural_range  rho_ood
 LiteJudge         0.0        1            0.999             0.852         0.930                   0.506               0.495          0.545      0.063    0.220     0.123    0.837          0.848         0.835         0.827                  0.841    0.747
 LiteJudge         0.3        1            1.000             0.989         0.985                   0.075               0.118          0.155      0.073    0.127     0.086    0.836          0.856         0.822         0.830                  0.839    0.748
  ImageCNN         0.0        1            0.991             0.711         0.855                   0.620               0.528          0.543      0.101    0.297     0.156    0.812          0.851         0.735         0.850                 

## 6. Figures

In [9]:
lab = {0.0: "natural only", MIX: f"+{int(MIX*100)}% PDSG"}
fig, ax = plt.subplots(1, 4, figsize=(22, 4.6))
for a, col, t in zip(ax, ["acc_lj_pdsg_test", "cue_share_n03_main", "acc_n03_main", "acc_ood"],
                     ["Accuracy, decoupled test pairs", "Cue share on VLM eval pairs (lower = less shortcut)", "Accuracy, VLM eval pairs (N03 main)", "Accuracy, out-of-distribution drawings"]):
    pv = JM.pivot(index="judge", columns="pdsg_share", values=col).reindex([j for j in order if j in JM.judge.values])
    pv.columns = [lab.get(c, c) for c in pv.columns]; pv.plot.bar(ax=a, rot=25, width=0.75); a.set_title(t, fontsize=10); a.set_xlabel("")
    if col.startswith("acc"): a.axhline(0.5, ls=":", c="grey"); a.set_ylim(0.3, 1.0)
plt.tight_layout(); plt.savefig(f"{OUTPUT_DIR}/fig_b_judges.png", dpi=130); plt.show()

bins = [0.02, 0.05, 0.1, 0.2, 0.4, 1.0]; dd = np.abs(OODP.d_ksm.values); bi = np.digitize(dd, bins) - 1
fig, ax = plt.subplots(1, 2, figsize=(14, 4.3))
for k in [k for k in SCORES if k.split("|")[0] in ("LiteJudge", "ImageCNN", "CLIPprobe", "LayoutMLP") and (len(k.split("|")) < 3 or k.endswith("s0"))]:
    a_ = [acc(UO[k][bi == i], OODP.d_ksm.values[bi == i]) for i in range(len(bins) - 1)]
    ax[0].plot(range(len(bins) - 1), a_, marker="o", ls="-" if float(k.split("|")[1]) > 0 else "--", label=k.replace("|", " p=").replace("p=0.0", "natural").split("|")[0])
ax[0].set_xticks(range(len(bins) - 1)); ax[0].set_xticklabels([f"{bins[i]}–{bins[i+1]}" for i in range(len(bins) - 1)]); ax[0].axhline(0.5, ls=":", c="grey")
ax[0].set_xlabel("|Δstress|"); ax[0].set_title("OOD accuracy by stress difference (dashed = natural only)"); ax[0].legend(fontsize=7)
o = OS[(OS.layout_pair_kind == "all") & OS.judge.str.startswith("LiteJudge")].pivot(index="style", columns="judge", values="acc")
o.plot.bar(ax=ax[1], rot=0); ax[1].set_ylim(0.4, 1); ax[1].axhline(0.5, ls=":", c="grey"); ax[1].set_title("LiteJudge OOD accuracy by rendering style")
plt.tight_layout(); plt.savefig(f"{OUTPUT_DIR}/fig_b_ood.png", dpi=130); plt.show()

if "LiteJudge|0.0" in SCORES and "LiteJudge|0.3" in SCORES:
    fig, ax = plt.subplots(1, 2, figsize=(13, 4.8), sharey=False)
    for a, k in zip(ax, ["LiteJudge|0.0", "LiteJudge|0.3"]):
        d = OD.assign(s=SCORES[k][1]); d = d[d["style"] == "train"]
        d["s_z"] = d.groupby("gname").s.transform(lambda v: (v - v.mean()) / (v.std() + 1e-9))
        for ln, g in d.groupby("layout"): a.scatter(g.ksm, g.s_z, s=14, label=ln)
        a.set_xlabel("true stress"); a.set_ylabel("LiteJudge score (z within graph)"); a.set_title(f"{k}: OOD drawings, training style")
    ax[1].legend(fontsize=7, ncol=2); plt.tight_layout(); plt.savefig(f"{OUTPUT_DIR}/fig_b_ood_scatter.png", dpi=130); plt.show()

g0 = list(OG)[0]; ex = OD[OD.gname == g0]
fig, ax = plt.subplots(3, 10, figsize=(22, 7))
for i, st in enumerate(OOD_STYLES):
    for j, (ln, r) in enumerate(ex[ex["style"] == st].set_index("layout").iterrows()):
        ax[i, j].imshow(OOD_IMG[int(r.oidx)], cmap="gray", vmin=0, vmax=255); ax[i, j].set_xticks([]); ax[i, j].set_yticks([])
        if i == 0: ax[i, j].set_title(f"{ln}\nstress {r.ksm:.2f}", fontsize=8)
    ax[i, 0].set_ylabel(st)
plt.suptitle(f"OOD examples: {g0}"); plt.tight_layout(); plt.savefig(f"{OUTPUT_DIR}/fig_b_ood_examples.png", dpi=110); plt.show()

## 7. Report and manifest

In [10]:
REP = {"runtime_min": round((time.time() - T0) / 60, 1), "judges": sorted(SCORES), "ood": {"graphs": list(OG), "drawings": len(OD), "pairs": len(OODP),
       "stress_min": round(float(OD.ksm.min()), 3), "stress_max": round(float(OD.ksm.max()), 3)},
       "table": JM[[c for c in show if c in JM]].round(4).to_dict("records"),
       "note": "LayoutMLP averaged over seeds; other judges seed 0. Accuracy on pairs with |Δstress| >= 0.02. pdsg_share 0 = natural only."}
json.dump(REP, open(f"{OUTPUT_DIR}/n10b_report.json", "w"), indent=1, default=str)
shutil.rmtree(f"{OUTPUT_DIR}/__pycache__", ignore_errors=True)
lines = [f"N10b baselines + OOD · {REP['runtime_min']} min · {len(SCORES)} judges · OOD {len(OODP)} pairs", ""]
for f in sorted(os.listdir(OUTPUT_DIR)):
    p = f"{OUTPUT_DIR}/{f}"
    if os.path.isfile(p): lines.append(f"{f:28s} {os.path.getsize(p)/1024:10.1f} KB")
open(f"{OUTPUT_DIR}/MANIFEST.txt", "w").write("\n".join(lines)); print("\n".join(lines))

N10b baselines + OOD · 21.7 min · 15 judges · OOD 2160 pairs

b_judges.csv                        2.9 KB
b_judges_seeds.csv                  3.7 KB
b_ood_by_style.csv                  3.8 KB
b_ood_pairs.parquet               267.2 KB
b_scores.parquet                   64.7 KB
fig_b_judges.png                   67.2 KB
fig_b_ood.png                     114.7 KB
fig_b_ood_examples.png            864.4 KB
fig_b_ood_scatter.png              76.5 KB
lj_train.py                         9.8 KB
n10b_report.json                    6.0 KB
pdcp_utils.py                      22.4 KB
